In [14]:
import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extras import execute_values
from pgvector.psycopg2 import register_vector
import os
import re
from dotenv import load_dotenv

load_dotenv()
conn = psycopg2.connect(os.getenv("DATABASE_URL"))
register_vector(conn)
cursor = conn.cursor()

print("Connected and pgvector type registered")

Connected and pgvector type registered


In [11]:
LOCAL_PATH = '../dataset/processed'

embeddings = np.load(f'{LOCAL_PATH}/asrs_embeddings.npy')
chunks_df = pd.read_csv(f'{LOCAL_PATH}/asrs_chunks_metadata.csv')

print(f"Loaded {len(chunks_df)} chunk records and {embeddings.shape[0]} embeddings")
assert len(chunks_df) == embeddings.shape[0], "Mismatch between metadata rows and embedding count"

Loaded 18886 chunk records and 18886 embeddings


In [12]:
# Batch insert into the chunks table

records = [
    (
        row['chunk_id'], str(row['acn']), row['text'], row['chunk_type'],
        row['aircraft_model'], row['flight_phase'], row['mission'],
        row['aircraft_operator'], row['airport'], row['state'],
        int(row['report_year']) if pd.notna(row['report_year']) else None,
        embeddings[i]
    )
    for i, row in chunks_df.iterrows()
]

execute_values(cursor, """
    INSERT INTO chunks (chunk_id, acn, text, chunk_type, aircraft_model, flight_phase,
                         mission, aircraft_operator, airport, state, report_year, embedding)
    VALUES %s
    ON CONFLICT (chunk_id) DO NOTHING
""", records)

conn.commit()
print(f"Inserted {cursor.rowcount} rows")

Inserted 0 rows


In [13]:
# Verify the data landed correctly
cursor.execute("SELECT COUNT(*) FROM chunks;")
print(f"Total rows in chunks table: {cursor.fetchone()[0]}")

cursor.execute("SELECT chunk_type, COUNT(*) FROM chunks GROUP BY chunk_type;")
print(cursor.fetchall())

Total rows in chunks table: 18886
[('synopsis', 9443), ('narrative', 9443)]


In [15]:
STOPWORDS = {'a', 'an', 'and', 'the', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
             'of', 'with', 'by', 'from', 'during', 'is', 'are', 'was', 'were'}

def build_keyword_query(text: str) -> str:
    words = re.findall(r'\w+', text.lower())
    filtered = [w for w in words if w not in STOPWORDS]
    return ' | '.join(filtered)

query_text = "engine vibration and bearing wear during climb"
or_query = build_keyword_query(query_text)
print(f"Cleaned OR query: {or_query}")

cursor.execute("""
    SELECT chunk_id, text, ts_rank(text_search_vector, query) AS keyword_rank
    FROM chunks, to_tsquery('english', %s) query
    WHERE text_search_vector @@ query
    ORDER BY keyword_rank DESC
    LIMIT 5;
""", (or_query,))

print("\nKeyword search results:")
for row in cursor.fetchall():
    print(f"{row[0]} (rank={row[2]:.4f}): {row[1][:150]}...")

Cleaned OR query: engine | vibration | bearing | wear | climb

Keyword search results:
1866115.0_narrative (rank=0.0638): Preflight; runup and take off were normal. At about 1;700 ft. I started smelling burnt rubber; I looked visually at the cowling and tires and saw noth...
947764.0_narrative (rank=0.0605): Approximately 10 minutes after takeoff; during climb; we started to hear repeated low frequency banging noises from the back of the aircraft; but didn...
1104659.0_narrative (rank=0.0603): After a normal takeoff; at approximately 1;000 feet; the First Officer and I noticed a slight vibration. At 1;500 feet when we set climb power to 1;05...
1379278.0_narrative (rank=0.0596): Aircraft departed on time. Weather was VFR 3 knot winds out of the southwest. We taxied out to runway 23 for departure. All aircraft indications were ...
1080539.0_narrative (rank=0.0585): After a normal pre-flight and taxi out the aircraft was ready for takeoff. I was the Captain and pilot not flying for the f

In [ ]:
# # Test hybrid retrieval end-to-end
# query_text = "engine vibration and bearing wear during climb"
# query_embedding = None  # placeholder — see note below

# cursor.execute("""
#     SELECT chunk_id, text, ts_rank(text_search_vector, query) AS keyword_rank
#     FROM chunks, plainto_tsquery('english', %s) query
#     WHERE text_search_vector @@ query
#     ORDER BY keyword_rank DESC
#     LIMIT 5;
# """, (query_text,))

# print("Keyword search results:")
# for row in cursor.fetchall():
#     print(f"{row[0]}: {row[1][:150]}...")

Keyword search results:


In [7]:
cursor.execute("SELECT chunk_id, text_search_vector FROM chunks LIMIT 1;")
row = cursor.fetchone()
print(f"chunk_id: {row[0]}")
print(f"text_search_vector (first 200 chars): {str(row[1])[:200]}")

chunk_id: 818235.0_narrative
text_search_vector (first 200 chars): '000':4,95 '10':94 '15':155 '16':3 '35':192 'acar':107 'action':62 'advis':78 'aft':42,164 'airport':75 'atc':76 'attend':148 'bay':16 'brief':150 'brought':29 'call':55,63 'cargo':43,165 'checklist':


In [8]:
def build_or_tsquery(text):
    words = text.lower().split()
    return ' | '.join(words)

query_text = "engine vibration and bearing wear during climb"
or_query = build_or_tsquery(query_text)
print(f"OR query string: {or_query}")

cursor.execute("""
    SELECT chunk_id, text, ts_rank(text_search_vector, query) AS keyword_rank
    FROM chunks, to_tsquery('english', %s) query
    WHERE text_search_vector @@ query
    ORDER BY keyword_rank DESC
    LIMIT 5;
""", (or_query,))

print("\nKeyword search results (OR logic):")
for row in cursor.fetchall():
    print(f"{row[0]} (rank={row[2]:.4f}): {row[1][:150]}...")

OR query string: engine | vibration | and | bearing | wear | during | climb

Keyword search results (OR logic):
1866115.0_narrative (rank=0.0638): Preflight; runup and take off were normal. At about 1;700 ft. I started smelling burnt rubber; I looked visually at the cowling and tires and saw noth...
947764.0_narrative (rank=0.0605): Approximately 10 minutes after takeoff; during climb; we started to hear repeated low frequency banging noises from the back of the aircraft; but didn...
1104659.0_narrative (rank=0.0603): After a normal takeoff; at approximately 1;000 feet; the First Officer and I noticed a slight vibration. At 1;500 feet when we set climb power to 1;05...
1379278.0_narrative (rank=0.0596): Aircraft departed on time. Weather was VFR 3 knot winds out of the southwest. We taxied out to runway 23 for departure. All aircraft indications were ...
1080539.0_narrative (rank=0.0585): After a normal pre-flight and taxi out the aircraft was ready for takeoff. I was the Captain and p

In [9]:
cursor.execute("""
    SELECT chunk_id, text, ts_rank(text_search_vector, query) AS keyword_rank
    FROM chunks, websearch_to_tsquery('english', %s) query
    WHERE text_search_vector @@ query
    ORDER BY keyword_rank DESC
    LIMIT 5;
""", (query_text,))

print("Keyword search results (websearch_to_tsquery):")
for row in cursor.fetchall():
    print(f"{row[0]} (rank={row[2]:.4f}): {row[1][:150]}...")

Keyword search results (websearch_to_tsquery):


In [ ]:
# Close connection

cursor.close()
conn.close()
print("Ingestion complete")